In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as img
import os
import cv2

In [2]:
img.imread('/kaggle/input/competitions/cat-and-dog-classification-harper2022/Cat/0.jpg')

array([[[203, 164,  87],
        [203, 164,  87],
        [204, 165,  88],
        ...,
        [240, 201, 122],
        [239, 200, 121],
        [238, 199, 120]],

       [[203, 164,  87],
        [203, 164,  87],
        [204, 165,  88],
        ...,
        [240, 201, 122],
        [239, 200, 121],
        [239, 200, 121]],

       [[203, 164,  87],
        [203, 164,  87],
        [204, 165,  88],
        ...,
        [241, 202, 123],
        [240, 201, 122],
        [239, 200, 121]],

       ...,

       [[153, 122,  55],
        [153, 122,  55],
        [153, 122,  55],
        ...,
        [  2,   2,   0],
        [  2,   2,   0],
        [  2,   2,   0]],

       [[152, 121,  54],
        [152, 121,  54],
        [152, 121,  54],
        ...,
        [  1,   1,   0],
        [  1,   1,   0],
        [  1,   1,   0]],

       [[151, 120,  53],
        [151, 120,  53],
        [152, 121,  54],
        ...,
        [  1,   1,   0],
        [  1,   1,   0],
        [  1,   1,   0]]

In [20]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

In [21]:
X_cats = []
y_cats = []
for i in os.listdir('/kaggle/input/competitions/cat-and-dog-classification-harper2022/Cat')[:1000]:
    try:
        image = img.imread(f'/kaggle/input/competitions/cat-and-dog-classification-harper2022/Cat/{i}')
        image = cv2.resize(image, (224,224))
        if len(image.shape) == 2:
            image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        image = image[:,:,:3]
        image = preprocess_input(image) 
        X_cats.append(image)
        y_cats.append(0)
    except:
        print(i)
X_cats = np.array(X_cats, dtype='float32')
y_cats = np.array(y_cats, dtype='float32')

In [22]:
X_dogs = []
y_dogs = []
for i in os.listdir('/kaggle/input/competitions/cat-and-dog-classification-harper2022/Dog')[:1000]:
    try:
        image = img.imread(f'/kaggle/input/competitions/cat-and-dog-classification-harper2022/Dog/{i}')
        image = cv2.resize(image, (224,224))
        if len(image.shape) == 2:
            image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        image = image[:,:,:3]
        image = preprocess_input(image) 
        X_dogs.append(image)
        y_dogs.append(1)
    except:
        print(i)
X_dogs = np.array(X_dogs, dtype='float32')
y_dogs = np.array(y_dogs, dtype='float32')

/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


In [25]:
X_cats.shape, y_cats.shape

((1000, 224, 224, 3), (1000,))

In [26]:
X_dogs.shape, y_dogs.shape

((1000, 224, 224, 3), (1000,))

In [27]:
X_train = np.concatenate([X_cats,X_dogs], axis=0)
y_train = np.concatenate([y_cats,y_dogs], axis=0)

In [28]:
from sklearn.utils import shuffle

In [29]:
X_train,y_train = shuffle(X_train,y_train, random_state=42)

In [30]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2
import tensorflow as tf
from keras.optimizers import Adam

In [33]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))

In [34]:
base_model.trainable=False

In [35]:
model = Sequential([
            base_model, 
            GlobalAveragePooling2D(), 
            Dense(1, activation='sigmoid')
])

In [36]:
model.compile(optimizer='Adam', loss='binary_crossentropy', metrics=['accuracy'])

In [37]:
model.fit(X_train,y_train, validation_split=0.15, epochs=10)

Epoch 1/10
54/54 ━━━━━━━━━━━━━━━━━━━━ 34s 559ms/step - accuracy: 0.8357 - loss: 0.3827 - val_accuracy: 0.9833 - val_loss: 0.0803
Epoch 2/10
54/54 ━━━━━━━━━━━━━━━━━━━━ 28s 519ms/step - accuracy: 0.9794 - loss: 0.0795 - val_accuracy: 0.9800 - val_loss: 0.0578
Epoch 3/10
54/54 ━━━━━━━━━━━━━━━━━━━━ 28s 516ms/step - accuracy: 0.9915 - loss: 0.0495 - val_accuracy: 0.9833 - val_loss: 0.0494
Epoch 4/10
54/54 ━━━━━━━━━━━━━━━━━━━━ 29s 539ms/step - accuracy: 0.9881 - loss: 0.0470 - val_accuracy: 0.9833 - val_loss: 0.0448
Epoch 5/10
54/54 ━━━━━━━━━━━━━━━━━━━━ 29s 532ms/step - accuracy: 0.9946 - loss: 0.0317 - val_accuracy: 0.9867 - val_loss: 0.0418
Epoch 6/10
54/54 ━━━━━━━━━━━━━━━━━━━━ 28s 520ms/step - accuracy: 0.9930 - loss: 0.0394 - val_accuracy: 0.9867 - val_loss: 0.0409
Epoch 7/10
54/54 ━━━━━━━━━━━━━━━━━━━━ 29s 540ms/step - accuracy: 0.9961 - loss: 0.0246 - val_accuracy: 0.9867 - val_loss: 0.0397
Epoch 8/10
54/54 ━━━━━━━━━━━━━━━━━━━━ 31s 573ms/step - accuracy: 0.9989 - loss: 0.0189 - val_accu

In [ ]:
y_pred = model.predict(X_test)